# Projet IC2 (1/3) — Dénoyage d’une fosse : données et variables retardées

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic2/notebooks/IC2_Projet_S3_donnees.ipynb)
> Données : dossier `IC2_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn, torch, shap.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['denoyage_fosse.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic2/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
pit = pd.read_csv('denoyage_fosse.csv', parse_dates=['semaine']); print(pit.describe().round(1))
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].bar(pit.semaine, pit.pluie_mm, width=6); ax[1].plot(pit.semaine, pit.pompage_m3h)
for c in ['P1_fosse', 'P2_150m', 'P3_600m']: ax[2].plot(pit.semaine, pit[c], label=c)
ax[2].legend(); plt.show()

## Variables retardées (sans fuite) et cible Δh à t + 4 semaines

In [ ]:
pit = pd.read_csv('denoyage_fosse.csv', parse_dates=['semaine'])
f = pd.DataFrame(index=pit.index)
for c in ['P1_fosse', 'P2_150m', 'P3_600m']:
    for lag in (0, 1, 2, 4): f[f'{c}_l{lag}'] = pit[c].shift(lag)
f['pluie_4s'] = pit['pluie_mm'].rolling(4).sum(); f['pluie_12s'] = pit['pluie_mm'].rolling(12).sum()
f['pompage_t'] = pit['pompage_m3h']; f['pompage_plan_t4'] = pit['pompage_m3h'].shift(-4)
j = pit['semaine'].dt.dayofyear; f['sin_saison'] = np.sin(2*np.pi*j/365); f['cos_saison'] = np.cos(2*np.pi*j/365)
f['cible'] = pit['P1_fosse'].shift(-4); f['delta'] = f['cible'] - pit['P1_fosse']
f = f.dropna()
date = pit.loc[f.index, 'semaine']; tr, te = f[date < '2024-01-01'], f[date >= '2024-01-01']
X = [c for c in f.columns if c not in ('cible', 'delta')]
rmse = lambda a, b: np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2))
print(len(tr), 'semaines d’entraînement,', len(te), 'de test')
print(X)

## Questions
1. `pompage_plan_t4` utilise le futur : est-ce une fuite ? (indice : plan d’exploitation connu à l’avance)
2. Comparer la plage de `pompage_t` à l’entraînement et au test. Conséquence pour un modèle à base d’arbres ?

In [ ]:
print('pompage max entraînement :', tr['pompage_t'].max(), ' test :', te['pompage_t'].max())
print('niveau P1 min entraînement :', round(tr['cible'].min(), 2), ' test :', round(te['cible'].min(), 2))